In [2]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 886.4 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 26.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 115.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 88.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 59.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 116.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [3]:
# ── Cell 0: Drive Mount ────────────────────────────────────────────────────
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    BASE_DIR = '/content/drive/MyDrive/BAH/isro_hackathon_data'
except ImportError:
    IN_COLAB = False
    BASE_DIR = os.environ.get('ISRO_DATA_DIR', './isro_hackathon_data')
    os.makedirs(BASE_DIR, exist_ok=True)

print(f'Running on Colab : {IN_COLAB}')
print(f'Data directory   : {BASE_DIR}')

Mounted at /content/drive
Running on Colab : True
Data directory   : /content/drive/MyDrive/BAH/isro_hackathon_data


In [4]:
# ── Cell 1: Imports ────────────────────────────────────────────────────────
import os, gc, math, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import weight_norm

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print('XGBoost not found — install with: pip install xgboost')

import psutil

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')

Device : cuda
PyTorch: 2.5.1+cu121


In [20]:
# ── Cell 2: Config ─────────────────────────────────────────────────────────
SAVE_DIR   = '/content/drive/MyDrive/BAH/isro_hackathon_data'
OUTPUT_DIR = '/content/drive/MyDrive/BAH/isro_hackathon_data/forecasting_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
CKPT_PATH  = os.path.join(OUTPUT_DIR, 'deepflare_best.pt')

# ── Features — 7 SHEF channels + 6 flare-history channels ─────────────────
# History features are inspired by DeFN (Nishizuka & Sugiura 2018), whose
# strongest predictors are flare-history counts/recency, not flux alone.
# Cost: zero new data — computed entirely from flares_cache.csv, which we
# already have. They are static per-window (don't vary across the 60
# timesteps) but are broadcast across the window as constant channels so
# the architecture and every downstream cell needs zero changes.
# bg_flux_27d_norm: 27d ~ one solar rotation period — captures whether an
# active region's BASELINE flux level is elevated across rotations, distinct
# information from the within-window flux/derivative features above.
FEATURE_COLS = ['log_soft', 'log_hard', 'log_hardening',
                'dFs_dt', 'dFh_dt', 'Fs_ma5', 'd2Fs_dt2',
                'bg_flux_27d_norm']
HISTORY_FEATURE_NAMES = ['Cplus_count_6h', 'Cplus_count_24h',
                          'Mplus_count_24h', 'Mplus_count_48h',
                          'mins_since_last_Cplus', 'mins_since_last_Mplus']
HISTORY_LOOKBACK_MIN  = {'Cplus_count_6h': 360, 'Cplus_count_24h': 1440,
                          'Mplus_count_24h': 1440, 'Mplus_count_48h': 2880}
HISTORY_RECENCY_CAP_MIN = 4320.0   # cap "minutes since" at 72h
N_FEATURES   = len(FEATURE_COLS) + len(HISTORY_FEATURE_NAMES)  # 8 + 6 = 14
N_CLASSES    = 5
CLASS_NAMES  = ['A/bg', 'B', 'C', 'M', 'X']

# ── Window params ──────────────────────────────────────────────────────────
WINDOW  = 60
HORIZON = 30
STRIDE  = 60

# ── GOES class thresholds (log10 scale) ───────────────────────────────────
LOG_THRESHOLDS = [-7.0, -6.0, -5.0, -4.0]

# ── Interleaved temporal split — model must see Cycle 25 in training ──────
# Simple cutoff splits fail because train never sees Solar Cycle 25 maximum
# (2023-2024), causing massive val/test AUC gap. Interleaved split puts
# specific months from 2021+ into val/test while keeping the REST of
# 2021-2024 (including cycle maximum months) in training.
VAL_MONTHS  = [1, 5, 9]     # Jan, May, Sep
TEST_MONTHS = [3, 7, 11]    # Mar, Jul, Nov
SPLIT_START_YEAR = 2021     # before this, everything goes to train

# ── TCN architecture ───────────────────────────────────────────────────────
KERNEL_SIZE = 3
DILATIONS   = [1, 2, 4, 8, 16]
DROPOUT     = 0.20

# ── Loss weights ───────────────────────────────────────────────────────────
FOCAL_GAMMA  = 2.0    # higher gamma = more focus on hard examples
ALPHA_ASPIRE = 1.0
BETA_ASPIRE  = 0.5
GAMMA_ASPIRE = 0.8
W_FORECAST   = 0.60
W_NOWCAST    = 0.35
W_TTP        = 0.05

# ── Training ───────────────────────────────────────────────────────────────
BATCH_SIZE    = 256     # lower than 512 to save RAM
EPOCHS        = 80
LR            = 5e-5    # stable LR — prevents NaN divergence
LR_MIN        = 1e-7
WARMUP_EPOCHS = 3
PATIENCE      = 15
MC_PASSES     = 50

print('Config loaded.')
print(f'Features      : {N_FEATURES} — {FEATURE_COLS}')
print(f'Window/Horizon: {WINDOW}/{HORIZON} min | Stride: {STRIDE} min')
print(f'Split         : interleaved — val months {VAL_MONTHS}, test months {TEST_MONTHS}, from {SPLIT_START_YEAR}+')

Config loaded.
Features      : 13 — ['log_soft', 'log_hard', 'log_hardening', 'dFs_dt', 'dFh_dt', 'Fs_ma5', 'd2Fs_dt2']
Window/Horizon: 60/30 min | Stride: 60 min
Split         : interleaved — val months [1, 5, 9], test months [3, 7, 11], from 2021+


In [ ]:
# ── Cell 1b: Add bg_flux_27d_norm to goes_preprocessed.csv (run once) ──────
# 27 days ~= one solar rotation. This captures whether an active region's
# BASELINE flux is elevated across repeat rotations -- different signal
# from the within-window flux/derivative features already in use. Chunked
# to avoid loading the full CSV into RAM at once. Idempotent: skips if the
# column is already present (safe to re-run this cell).
import pandas as pd, numpy as np, os, gc

_csv_path = os.path.join(SAVE_DIR, 'goes_preprocessed.csv')
_cols = pd.read_csv(_csv_path, nrows=0).columns.tolist()

if 'bg_flux_27d_norm' not in _cols:
    print('Adding bg_flux_27d_norm (chunked, low-RAM)...')
    log_soft_series = pd.read_csv(_csv_path, usecols=['log_soft'])['log_soft']

    # 38880 = 27 days at 1-min cadence; min_periods=1440 (1 day) so early
    # rows still get a value instead of NaN.
    bg = log_soft_series.rolling(window=38880, min_periods=1440, center=False).median()
    bg = bg.fillna(log_soft_series.expanding(min_periods=1).median())
    bg_mean = bg.mean(); bg_std = bg.std() + 1e-8
    bg_norm = (bg - bg_mean) / bg_std

    bg_norm.to_csv(os.path.join(SAVE_DIR, 'bg_norm_temp.csv'),
                   index=False, header=['bg_flux_27d_norm'])
    del log_soft_series, bg, bg_norm; gc.collect()

    bg_col = pd.read_csv(os.path.join(SAVE_DIR, 'bg_norm_temp.csv'))['bg_flux_27d_norm']
    out_path = os.path.join(SAVE_DIR, 'goes_with_bg.csv')
    first, rows_done = True, 0
    for chunk in pd.read_csv(_csv_path, chunksize=500_000):
        end = rows_done + len(chunk)
        chunk['bg_flux_27d_norm'] = bg_col.iloc[rows_done:end].values
        chunk.to_csv(out_path, mode='w' if first else 'a', header=first, index=False)
        rows_done = end; first = False
    os.replace(out_path, _csv_path)
    os.remove(os.path.join(SAVE_DIR, 'bg_norm_temp.csv'))
    print(f'Done. Columns now: {pd.read_csv(_csv_path, nrows=0).columns.tolist()}')
else:
    print('bg_flux_27d_norm already present in goes_preprocessed.csv — skipping.')


In [21]:
import os
for f in ['X_windows_7ch.npy', 'y_5class.npy', 'y_nowcast.npy', 'y_ttp.npy', 'window_times_7ch.npy']:
    p = os.path.join(SAVE_DIR, f)
    if os.path.exists(p):
        os.remove(p)
        print(f'Deleted stale checkpoint: {f}')

Deleted stale checkpoint: X_windows_7ch.npy
Deleted stale checkpoint: y_5class.npy
Deleted stale checkpoint: y_nowcast.npy
Deleted stale checkpoint: y_ttp.npy
Deleted stale checkpoint: window_times_7ch.npy


In [36]:
# ── Cell 3: Load 7-Channel Windowed Dataset (mmap — minimal RAM) ───────────
print('Files available in SAVE_DIR:')
for f in sorted(os.listdir(SAVE_DIR)):
    p = os.path.join(SAVE_DIR, f)
    if os.path.isfile(p):
        print(f'  {f:45s} {os.path.getsize(p)/1e6:8.1f} MB')
    else:
        print(f'  {f}/  (folder)')

# Filenames bumped to *_13ch* (was *_7ch*) so the new history-augmented
# array is never confused with, or silently overwritten by, the old cache.
CHECKPOINT_X     = os.path.join(SAVE_DIR, 'X_windows_14ch.npy')  # bumped 13->14 (added bg_flux_27d_norm)
CHECKPOINT_Y5    = os.path.join(SAVE_DIR, 'y_5class.npy')
CHECKPOINT_YNC   = os.path.join(SAVE_DIR, 'y_nowcast.npy')
CHECKPOINT_YTTP  = os.path.join(SAVE_DIR, 'y_ttp.npy')
CHECKPOINT_TIMES = os.path.join(SAVE_DIR, 'window_times_14ch.npy')

_checkpoint_files = [CHECKPOINT_X, CHECKPOINT_Y5, CHECKPOINT_YNC,
                     CHECKPOINT_YTTP, CHECKPOINT_TIMES]

if all(os.path.exists(p) for p in _checkpoint_files):
    print('\nLoading existing 13-channel checkpoint (mmap — does not load into RAM yet)...')
    X            = np.load(CHECKPOINT_X, mmap_mode='r')
    y_5class     = np.load(CHECKPOINT_Y5)
    y_nowcast    = np.load(CHECKPOINT_YNC)
    y_ttp        = np.load(CHECKPOINT_YTTP)
    window_times = np.load(CHECKPOINT_TIMES, allow_pickle=True)

    print(f'X shape        : {X.shape}   (mmap — on disk, not in RAM)')
    print(f'y_5class shape : {y_5class.shape}')
    print(f'window_times   : {window_times[0]} -> {window_times[-1]}')

    assert X.shape[2] == N_FEATURES, (
        f'X has {X.shape[2]} features but N_FEATURES={N_FEATURES}. '
        f'Check FEATURE_COLS in Cell 2.')
    print(f'Feature count check: {X.shape[2]} == {N_FEATURES} OK')

else:
    missing = [p for p in _checkpoint_files if not os.path.exists(p)]
    print(f'\nCheckpoint files missing: {missing}')
    print('Rebuilding the 14-channel windowed dataset (8 SHEF + 6 flare-history) '
          'from goes_preprocessed.csv and flares_cache.csv. This path '
          'pre-allocates arrays (instead of a growing Python list) specifically '
          'to avoid the RAM crash seen when this cell rebuilds from scratch.')

    def log_soft_to_class(v):
        if   v < LOG_THRESHOLDS[0]: return 0
        elif v < LOG_THRESHOLDS[1]: return 1
        elif v < LOG_THRESHOLDS[2]: return 2
        elif v < LOG_THRESHOLDS[3]: return 3
        else:                       return 4

    def class_char_to_idx(s):
        if not s or not isinstance(s, str):
            return None
        c = s.strip()[0].upper()
        return {'A':0,'B':1,'C':2,'M':3,'X':4}.get(c, None)

    def _count_windows(goes_df, window, horizon, stride):
        total = 0
        for _, seg_df in goes_df.groupby('segment_id', sort=True):
            n = len(seg_df)
            if n < window + horizon:
                continue
            total += len(range(0, n - window - horizon, stride))
        return total

    def build_windows_full(goes_df, flares_df, feature_cols,
                            window=60, horizon=30, stride=5):
        mx_mask  = flares_df['class'].str[:1].isin(['M', 'X'])
        all_mask = flares_df['class'].notna() & (flares_df['class'].str.strip() != '')

        mx_df  = flares_df[mx_mask].copy()
        all_df = flares_df[all_mask].copy()
        all_df['class_idx'] = all_df['class'].apply(class_char_to_idx)
        all_df = all_df.dropna(subset=['class_idx'])
        all_df['class_idx'] = all_df['class_idx'].astype(int)

        time_col = 'peak_time' if 'peak_time' in flares_df.columns else 'start_time'

        all_df_sorted = all_df.sort_values('start_time').reset_index(drop=True)
        all_times_ns  = all_df_sorted['start_time'].values.astype('int64')
        all_cls_arr   = all_df_sorted['class_idx'].values

        mx_df_sorted = mx_df.sort_values('start_time').reset_index(drop=True)
        mx_times_ns  = mx_df_sorted['start_time'].values.astype('int64')
        mx_peak_ns   = mx_df_sorted[time_col].values.astype('int64')

        ns_per_min = 60 * 1_000_000_000

        # ── Flare-history feature helper (DeFN-inspired) ───────────────
        # Causal only: looks strictly BEFORE w_end_ns, no leakage into the
        # forecast horizon. C+ = class_idx>=2, M+ = class_idx>=3.
        n_hist = len(HISTORY_FEATURE_NAMES)
        def history_features(w_end_ns):
            idx_before = np.searchsorted(all_times_ns, w_end_ns, side='left')
            past_cls   = all_cls_arr[:idx_before]
            past_t     = all_times_ns[:idx_before]
            row = np.zeros(n_hist, dtype=np.float32)
            row[0] = ((past_cls >= 2) &
                       (past_t >= w_end_ns - HISTORY_LOOKBACK_MIN['Cplus_count_6h']*ns_per_min)).sum()
            row[1] = ((past_cls >= 2) &
                       (past_t >= w_end_ns - HISTORY_LOOKBACK_MIN['Cplus_count_24h']*ns_per_min)).sum()
            row[2] = ((past_cls >= 3) &
                       (past_t >= w_end_ns - HISTORY_LOOKBACK_MIN['Mplus_count_24h']*ns_per_min)).sum()
            row[3] = ((past_cls >= 3) &
                       (past_t >= w_end_ns - HISTORY_LOOKBACK_MIN['Mplus_count_48h']*ns_per_min)).sum()
            c_mask = past_cls >= 2
            m_mask = past_cls >= 3
            row[4] = (min((w_end_ns - past_t[c_mask][-1]) / ns_per_min, HISTORY_RECENCY_CAP_MIN)
                       if c_mask.any() else HISTORY_RECENCY_CAP_MIN)
            row[5] = (min((w_end_ns - past_t[m_mask][-1]) / ns_per_min, HISTORY_RECENCY_CAP_MIN)
                       if m_mask.any() else HISTORY_RECENCY_CAP_MIN)
            return row

        n_total  = _count_windows(goes_df, window, horizon, stride)
        n_ch_out = len(feature_cols) + n_hist
        print(f'  Pre-allocating for {n_total:,} windows '
              f'({n_total * window * n_ch_out * 4 / 1e9:.2f} GB)...')

        X_out    = np.empty((n_total, window, n_ch_out), dtype=np.float32)
        y5_out   = np.empty(n_total, dtype=np.int64)
        ync_out  = np.empty(n_total, dtype=np.int64)
        yttp_out = np.empty(n_total, dtype=np.float32)
        wt_out   = np.empty(n_total, dtype='datetime64[ns]')

        write_i = 0
        segments = list(goes_df.groupby('segment_id', sort=True))
        n_segs   = len(segments)

        for seg_num, (seg_id, seg_df) in enumerate(segments):
            seg_df = seg_df.reset_index(drop=True)
            n_rows = len(seg_df)
            if n_rows < window + horizon:
                continue

            values   = seg_df[feature_cols].values.astype(np.float32)
            t_arr    = seg_df['time'].values
            t_arr_ns = t_arr.astype('int64')

            for i in range(0, n_rows - window - horizon, stride):
                w_end_ns = t_arr_ns[i + window]
                h_end_ns = w_end_ns + horizon * ns_per_min

                lo = np.searchsorted(all_times_ns, w_end_ns, side='right')
                hi = np.searchsorted(all_times_ns, h_end_ns, side='right')
                y5 = int(all_cls_arr[lo:hi].max()) if hi > lo else 0

                last_log_soft = values[i + window - 1, 0]
                ync = log_soft_to_class(last_log_soft)

                lo_mx = np.searchsorted(mx_times_ns, w_end_ns, side='right')
                if lo_mx < len(mx_peak_ns):
                    delta_min = (mx_peak_ns[lo_mx] - w_end_ns) / ns_per_min
                    yttp = float(delta_min) if 0.0 < delta_min <= 90.0 else 0.0
                else:
                    yttp = 0.0

                hist_row = history_features(w_end_ns)

                X_out[write_i, :, :len(feature_cols)] = values[i:i + window]
                X_out[write_i, :, len(feature_cols):]  = hist_row  # broadcast static across time
                y5_out[write_i]    = y5
                ync_out[write_i]   = ync
                yttp_out[write_i]  = yttp
                wt_out[write_i]    = t_arr[i + window]
                write_i += 1

            if seg_num % 200 == 0:
                print(f'  Segment {seg_num}/{n_segs} | windows written: {write_i:,}')

        X_out, y5_out, ync_out, yttp_out, wt_out = (
            X_out[:write_i], y5_out[:write_i], ync_out[:write_i],
            yttp_out[:write_i], wt_out[:write_i])
        return X_out, y5_out, ync_out, yttp_out, wt_out

    goes = pd.read_csv(os.path.join(SAVE_DIR, 'goes_preprocessed.csv'))
    goes['time'] = pd.to_datetime(goes['time'], format='mixed')
    goes = goes.sort_values('time').reset_index(drop=True)
    print(f'GOES data: {len(goes)} rows | segments: {goes["segment_id"].nunique()}')

    missing_feat = [c for c in FEATURE_COLS if c not in goes.columns]
    if missing_feat:
        raise ValueError(
            f'Missing SHEF features in goes_preprocessed.csv: {missing_feat}\n'
            'Run the SHEF feature block in the preprocessing notebook first.')

    flares = pd.read_csv(os.path.join(SAVE_DIR, 'flares_cache.csv'))
    flares['start_time'] = pd.to_datetime(flares['start_time'], format='mixed')
    if 'peak_time' in flares.columns:
        flares['peak_time'] = pd.to_datetime(flares['peak_time'], format='mixed')
    else:
        flares['peak_time'] = flares['start_time']
    print(f'Flares catalog: {len(flares)} events')

    t0 = time.time()
    X, y_5class, y_nowcast, y_ttp, window_times = build_windows_full(
        goes, flares, FEATURE_COLS, window=WINDOW, horizon=HORIZON, stride=STRIDE)
    print(f'Done in {(time.time()-t0)/60:.1f} min')

    np.save(CHECKPOINT_X,     X)
    np.save(CHECKPOINT_Y5,    y_5class)
    np.save(CHECKPOINT_YNC,   y_nowcast)
    np.save(CHECKPOINT_YTTP,  y_ttp)
    np.save(CHECKPOINT_TIMES, window_times)
    print(f'Saved 13-channel checkpoint to {SAVE_DIR}')

    del goes
    gc.collect()

    X = np.load(CHECKPOINT_X, mmap_mode='r')

CLASS_NAMES = ['A/bg', 'B', 'C', 'M', 'X']
counts      = np.bincount(y_5class, minlength=5)
total       = len(y_5class)
mx_positive = (y_5class >= 3).sum()
print(f'\nDataset summary')
print(f'  Total windows : {total:,}')
print(f'  M/X positive  : {mx_positive:,} ({mx_positive/total*100:.3f}%)')
print(f'  Class counts  : ' +
      ' | '.join(f'{n}: {c:,} ({c/total*100:.2f}%)'
                 for n, c in zip(CLASS_NAMES, counts)))

ram = psutil.virtual_memory()
print(f'RAM used: {ram.used/1e9:.1f} GB / {ram.total/1e9:.1f} GB ({ram.percent:.1f}%)')

Files available in SAVE_DIR:
  X_test.npy                                       850.8 MB
  X_train.npy                                     8589.6 MB
  X_val.npy                                       1695.9 MB
  X_windows.npy                                  11136.3 MB
  X_windows_13ch.npy                               374.1 MB
  flare_plots/  (folder)
  flares_cache.csv                                   0.5 MB
  forecasting_outputs/  (folder)
  goes_14year_raw.csv                              265.6 MB
  goes_preprocessed.csv                           1398.9 MB
  master_catalogue.csv                               6.5 MB
  roc_curve.png                                      0.1 MB
  standard_scaler.pkl                                0.0 MB
  window_times.npy                                  53.0 MB
  window_times_13ch.npy                              1.0 MB
  y_5class.npy                                       1.0 MB
  y_labels.npy                                       6.6 MB
  y_nowcast.n

In [59]:
# ── Cell 4: Interleaved Train / Val / Test Split (memory-safe) ────────────
window_times_dt = pd.to_datetime(window_times)
years  = window_times_dt.year
months = window_times_dt.month

post_split_start = years >= SPLIT_START_YEAR
val_mask   = post_split_start & months.isin(VAL_MONTHS)
test_mask  = post_split_start & months.isin(TEST_MONTHS)
train_mask = ~val_mask & ~test_mask

train_idx = np.where(train_mask)[0]
val_idx   = np.where(val_mask)[0]
test_idx  = np.where(test_mask)[0]

assert len(train_idx) > 0 and len(val_idx) > 0 and len(test_idx) > 0, (
    'One of the splits is empty — check VAL_MONTHS/TEST_MONTHS/SPLIT_START_YEAR.')

print(f'Train windows: {len(train_idx):,}')
print(f'Val windows  : {len(val_idx):,}   (Jan/May/Sep from {SPLIT_START_YEAR}+)')
print(f'Test windows : {len(test_idx):,}  (Mar/Jul/Nov from {SPLIT_START_YEAR}+)')

print('\nMaterializing splits one at a time (bounds peak RAM)...')

X_train    = np.array(X[train_idx])
y5_train   = y_5class[train_idx]
ync_train  = y_nowcast[train_idx]
yttp_train = y_ttp[train_idx]
gc.collect()
print(f'  X_train materialized: {X_train.shape}  ({X_train.nbytes/1e9:.2f} GB)')

X_val    = np.array(X[val_idx])
y5_val   = y_5class[val_idx]
ync_val  = y_nowcast[val_idx]
yttp_val = y_ttp[val_idx]
gc.collect()
print(f'  X_val materialized  : {X_val.shape}  ({X_val.nbytes/1e9:.2f} GB)')

X_test    = np.array(X[test_idx])
y5_test   = y_5class[test_idx]
ync_test  = y_nowcast[test_idx]
yttp_test = y_ttp[test_idx]
gc.collect()
print(f'  X_test materialized : {X_test.shape}  ({X_test.nbytes/1e9:.2f} GB)')

print(f'\nM+ fraction — Train: {(y5_train>=3).mean()*100:.3f}% | '
      f'Val: {(y5_val>=3).mean()*100:.3f}% | '
      f'Test: {(y5_test>=3).mean()*100:.3f}%')

Train windows: 102,976
Val windows  : 8,432   (Jan/May/Sep from 2021+)
Test windows : 8,501  (Mar/Jul/Nov from 2021+)

Materializing splits one at a time (bounds peak RAM)...
  X_train materialized: (102976, 60, 13)  (0.32 GB)
  X_val materialized  : (8432, 60, 13)  (0.03 GB)
  X_test materialized : (8501, 60, 13)  (0.03 GB)

M+ fraction — Train: 0.437% | Val: 1.020% | Test: 0.682%


In [60]:
# ── Cell 5: Subsample + Standardise (stratified) ───────────────────────────
#
# Subsample: keep ALL M/X flare windows, keep ALL C-class (hard negatives —
# these look like flare buildup but didn't cross into M/X, exactly the
# boundary cases the model needs to see to sharpen its decision surface),
# and only lightly subsample A/B (easy, low-information negatives — quiet
# Sun, nothing to discriminate). A flat 40%-of-everything subsample was
# spending RAM budget equally on easy and hard negatives; this reallocates
# the budget toward the negatives that actually teach the boundary.

np.random.seed(42)

# Capture the TRUE (pre-subsample) nowcast class distribution before we
# distort it below — needed so nowcast class weights (Cell 6) correct for
# the real population, not the artificially C-heavy training sample.
ORIG_NOWCAST_COUNTS = np.bincount(ync_train, minlength=N_CLASSES).astype(float)

mx_idx    = np.where(y5_train >= 3)[0]                       # M/X — keep all
c_idx     = np.where(y5_train == 2)[0]                       # C — keep all (hard negatives)
ab_idx    = np.where(y5_train < 2)[0]                        # A/B — light subsample only
keep_ab   = np.random.choice(ab_idx,
                              size=int(len(ab_idx) * 0.25),
                              replace=False)
keep_idx  = np.sort(np.concatenate([mx_idx, c_idx, keep_ab]))

X_train    = X_train[keep_idx];   y5_train   = y5_train[keep_idx]
ync_train  = ync_train[keep_idx]; yttp_train = yttp_train[keep_idx]

print(f'After subsample: {X_train.shape}')
print(f'M+ windows kept: {(y5_train>=3).sum():,} (100% of all flares)')
print(f'M+ rate        : {(y5_train>=3).mean()*100:.3f}% (real base rate)')

# StandardScaler — fit on train, apply to all
# Global normalisation: model sees absolute flux differences across solar cycle
print('\nFitting StandardScaler on train...')
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(
    X_train.reshape(-1, N_FEATURES)).reshape(X_train.shape)
X_val_s   = scaler.transform(
    X_val.reshape(-1, N_FEATURES)).reshape(X_val.shape)
X_test_s  = scaler.transform(
    X_test.reshape(-1, N_FEATURES)).reshape(X_test.shape)

# Save scaler for deployment
np.save(os.path.join(OUTPUT_DIR, 'scaler_mean.npy'), scaler.mean_)
np.save(os.path.join(OUTPUT_DIR, 'scaler_std.npy'),  scaler.scale_)

# Free raw arrays
del X_train, X_val, X_test
gc.collect()

print('Scaler fit and saved.')
ram = psutil.virtual_memory()
print(f'RAM used: {ram.used/1e9:.1f} GB ({ram.percent:.1f}%)')

After subsample: (29590, 60, 13)
M+ windows kept: 450 (100% of all flares)
M+ rate        : 1.521% (real base rate)

Fitting StandardScaler on train...
Scaler fit and saved.
RAM used: 3.5 GB (28.0%)


In [61]:
# ── Cell 6: Class Weights ──────────────────────────────────────────────────
train_counts = np.bincount(y5_train, minlength=N_CLASSES).astype(float)
# sqrt-softened (was raw 1/freq) — same fix already applied to the nowcast
# head. Raw inverse-frequency gave extreme weight ratios (~1000x+) that
# likely contributed to the epoch-to-epoch TSS oscillation seen in training
# (0.45 -> 0.60 -> 0.45 swings) — a jagged loss landscape from one class
# being weighted orders of magnitude more than another. Forecast still
# needs to be MORE aggressive than nowcast (rare M/X really is the whole
# point here), so we keep alpha (focal) and beta (TSS-reg) doing the
# rare-class emphasis work, and let the class weights themselves be gentler.
inv_freq     = 1.0 / np.sqrt(train_counts + 1e-6)
inv_freq     = inv_freq / inv_freq.sum() * N_CLASSES
CLASS_WEIGHTS = torch.FloatTensor(inv_freq).to(DEVICE)

# ── Separate, more balanced weights for the NOWCAST head ───────────────────
# CLASS_WEIGHTS (above) is correctly aggressive for the forecast head, but
# reusing it for nowcast caused A/bg (35% of data, a legitimate common class)
# to be weighted 1350x lower than X — the model stopped predicting it.
# Nowcast needs gentler, more balanced weighting since all 5 classes are
# meaningful "what is happening right now" labels, not just rare-event alarms.

# Use the TRUE pre-subsample population counts (captured in Cell 5), not
# ync_train's post-subsample counts — the stratified subsample (keep all C,
# downsample A/B) was deliberately distorted to help the forecast task, but
# that same distortion was silently corrupting nowcast's class weights,
# causing nowcast accuracy to regress (0.82 -> 0.59, M recall 0.94 -> 0.13)
# when the forecast-side subsampling change was introduced.
nowcast_counts = ORIG_NOWCAST_COUNTS
nowcast_inv    = 1.0 / np.sqrt(nowcast_counts + 1e-6)   # sqrt softens the weighting
nowcast_inv    = nowcast_inv / nowcast_inv.sum() * N_CLASSES
NOWCAST_CLASS_WEIGHTS = torch.FloatTensor(nowcast_inv).to(DEVICE)

print(f"\nNowcast class weights (sqrt-softened): "
      f"{dict(zip(CLASS_NAMES, NOWCAST_CLASS_WEIGHTS.cpu().numpy().round(4)))}")

print('Class counts in training (after subsample):')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name}: {int(train_counts[i]):,} ({100*train_counts[i]/len(y5_train):.3f}%)')
print(f'\nClass weights: {dict(zip(CLASS_NAMES, CLASS_WEIGHTS.cpu().numpy().round(4)))}')
print('\nNO WeightedRandomSampler — imbalance handled by focal loss (gamma=3)')


Nowcast class weights (sqrt-softened): {'A/bg': np.float32(0.1365), 'B': np.float32(0.1257), 'C': np.float32(0.1728), 'M': np.float32(0.9166), 'X': np.float32(3.6484)}
Class counts in training (after subsample):
  A/bg: 23,730 (80.196%)
  B: 732 (2.474%)
  C: 4,678 (15.809%)
  M: 422 (1.426%)
  X: 28 (0.095%)

Class weights: {'A/bg': np.float32(0.1098), 'B': np.float32(0.6249), 'C': np.float32(0.2472), 'M': np.float32(0.823), 'X': np.float32(3.1951)}

NO WeightedRandomSampler — imbalance handled by focal loss (gamma=3)


In [62]:
# ── Cell 7: Dataset & DataLoaders ──────────────────────────────────────────
class SolarFlareDataset(Dataset):
    def __init__(self, X, yf, ync, yttp):
        self.X    = torch.FloatTensor(X)
        self.yf   = torch.LongTensor(yf)
        self.ync  = torch.LongTensor(ync)
        self.yttp = torch.FloatTensor(yttp)
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.yf[i], self.ync[i], self.yttp[i]

train_ds = SolarFlareDataset(X_train_s, y5_train, ync_train, yttp_train)
val_ds   = SolarFlareDataset(X_val_s,   y5_val,   ync_val,   yttp_val)
test_ds  = SolarFlareDataset(X_test_s,  y5_test,  ync_test,  yttp_test)

# shuffle=True for train, NO sampler
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE * 2,
                          shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE * 2,
                          shuffle=False, num_workers=0, pin_memory=False)

print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')
print(f'Real M+ rate in train batches: ~{(y5_train>=3).mean()*100:.3f}%')

ram = psutil.virtual_memory()
print(f'RAM used: {ram.used/1e9:.1f} GB ({ram.percent:.1f}%)')

Train batches: 116 | Val: 17 | Test: 17
Real M+ rate in train batches: ~1.521%
RAM used: 3.4 GB (27.8%)


In [63]:
# ── Cell 8: TCN Architecture ───────────────────────────────────────────────
class CausalResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.conv1 = weight_norm(nn.Conv1d(
            in_ch, out_ch, kernel_size,
            padding=pad, dilation=dilation))
        self.conv2 = weight_norm(nn.Conv1d(
            out_ch, out_ch, kernel_size,
            padding=pad, dilation=dilation))
        self.drop  = nn.Dropout(dropout)
        self.skip  = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.pad   = pad

    def forward(self, x):
        h = F.relu(self.conv1(x)[..., :-self.pad])
        h = self.drop(h)
        h = F.relu(self.conv2(h)[..., :-self.pad])
        h = self.drop(h)
        return F.relu(h + self.skip(x))


class DeepFlarePredictor(nn.Module):
    """
    Dual-task TCN:
      - Forecast head : P(flare class) in next HORIZON minutes
      - Nowcast head  : current GOES class
      - TTP head      : time to peak (regression)
    Input: (batch, window, n_features) — transposed to (batch, n_features, window)
    """
    def __init__(self, n_features=7, n_classes=5,
                 channels=None, dilations=None,
                 kernel_size=3, dropout=0.2):
        super().__init__()
        if channels  is None: channels  = [n_features, 32, 64, 64, 128, 128]
        if dilations is None: dilations = [1, 2, 4, 8, 16]

        # CRITICAL: first channel must equal n_features
        assert channels[0] == n_features, (
            f'channels[0]={channels[0]} must equal n_features={n_features}')

        blocks = []
        for i, d in enumerate(dilations):
            in_c  = channels[i]
            out_c = channels[i + 1]
            blocks.append(CausalResidualBlock(in_c, out_c, kernel_size, d, dropout))
        self.tcn = nn.Sequential(*blocks)

        enc_dim = channels[-1]
        # Forecast head
        self.forecast_head = nn.Sequential(
            nn.Linear(enc_dim, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_classes))
        # Nowcast head
        self.nowcast_head  = nn.Sequential(
            nn.Linear(enc_dim, 32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, n_classes))
        # TTP regression head
        self.ttp_head      = nn.Sequential(
            nn.Linear(enc_dim, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.ReLU())  # time >= 0

    def forward(self, x):
        # x: (B, T, C) → (B, C, T) for Conv1d
        h = self.tcn(x.permute(0, 2, 1))
        # Global max pool over time
        enc = h.max(dim=-1).values
        return self.forecast_head(enc), self.nowcast_head(enc), self.ttp_head(enc)


model = DeepFlarePredictor(
    n_features  = N_FEATURES,
    n_classes   = N_CLASSES,
    channels    = [N_FEATURES, 48, 96, 96, 160, 160],  # bumped — input grew 7->13ch
    dilations   = DILATIONS,
    kernel_size = KERNEL_SIZE,
    dropout     = DROPOUT
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {n_params:,}')
print(f'Input  : (batch, {WINDOW}, {N_FEATURES})')
print(f'Output : forecast(batch,5) | nowcast(batch,5) | ttp(batch,1)')

Model parameters: 426,299
Input  : (batch, 60, 13)
Output : forecast(batch,5) | nowcast(batch,5) | ttp(batch,1)


In [64]:
# ── Cell 9: ASPIRE Loss + Utilities ───────────────────────────────────────
class ASPIRELoss(nn.Module):
    """
    Asymmetric Skill-weighted Probabilistic Inference and Recall Enhancement.
    L = alpha*focal + beta*TSS_reg + gamma*Brier
    gamma=3 focuses aggressively on hard (M/X) examples.
    """
    def __init__(self, class_weights, n_classes=5,
                 focal_gamma=3.0,
                 alpha=1.0, beta=0.5, gamma_cal=0.3):
        super().__init__()
        self.n_classes   = n_classes
        self.focal_gamma = focal_gamma
        self.alpha       = alpha
        self.beta        = beta
        self.gamma_cal   = gamma_cal
        self.register_buffer('class_weights', class_weights)

    def forward(self, logits, targets):
        # Clamp logits to prevent inf/nan
        logits = torch.clamp(logits, -20, 20)
        probs  = F.softmax(logits, dim=1)
        n      = logits.size(0)

        # ── Focal loss ────────────────────────────────────────────────
        log_p  = F.log_softmax(logits, dim=1)
        p_t    = probs[range(n), targets]
        cw_t   = self.class_weights[targets]
        focal  = -cw_t * (1 - p_t) ** self.focal_gamma * log_p[range(n), targets]
        L_focal = focal.mean()

        # ── Differentiable TSS regularisation ─────────────────────────
        mx_mask  = (targets >= 3).float()
        p_mx     = probs[:, 3] + probs[:, 4]
        soft_tp  = (p_mx * mx_mask).sum()
        soft_fn  = ((1 - p_mx) * mx_mask).sum()
        soft_fp  = (p_mx * (1 - mx_mask)).sum()
        soft_tn  = ((1 - p_mx) * (1 - mx_mask)).sum()
        tpr_soft = soft_tp / (soft_tp + soft_fn + 1e-6)
        fpr_soft = soft_fp / (soft_fp + soft_tn + 1e-6)
        L_tss    = 1.0 - (tpr_soft - fpr_soft)

        # ── Brier calibration ─────────────────────────────────────────
        one_hot = F.one_hot(targets, self.n_classes).float()
        L_cal   = ((probs - one_hot) ** 2).sum(dim=1).mean()

        total = self.alpha * L_focal + self.beta * L_tss + self.gamma_cal * L_cal
        return total, {
            'focal': L_focal.item(),
            'tss'  : L_tss.item(),
            'cal'  : L_cal.item()
        }


aspire_loss = ASPIRELoss(CLASS_WEIGHTS, N_CLASSES,
                          focal_gamma=FOCAL_GAMMA,
                          alpha=ALPHA_ASPIRE,
                          beta=BETA_ASPIRE,
                          gamma_cal=GAMMA_ASPIRE).to(DEVICE)

focal_only  = ASPIRELoss(NOWCAST_CLASS_WEIGHTS, N_CLASSES,
                          focal_gamma=FOCAL_GAMMA,
                          alpha=1.0, beta=0.0, gamma_cal=0.0).to(DEVICE)

ttp_loss_fn = nn.HuberLoss(delta=10.0)

print('ASPIRE loss built.')
print(f'  focal_gamma={FOCAL_GAMMA} | alpha={ALPHA_ASPIRE} | '
      f'beta={BETA_ASPIRE} | gamma_cal={GAMMA_ASPIRE}')

ASPIRE loss built.
  focal_gamma=2.0 | alpha=1.0 | beta=0.5 | gamma_cal=0.8


In [65]:
# ── Cell 10: Evaluation Utilities ──────────────────────────────────────────
def compute_metrics(y_true_binary, y_prob_mx, threshold=0.5, label=''):
    y_pred = (y_prob_mx >= threshold).astype(int)
    TP = int(((y_pred==1)&(y_true_binary==1)).sum())
    FP = int(((y_pred==1)&(y_true_binary==0)).sum())
    FN = int(((y_pred==0)&(y_true_binary==1)).sum())
    TN = int(((y_pred==0)&(y_true_binary==0)).sum())
    TPR = TP/(TP+FN+1e-9); FPR = FP/(FP+TN+1e-9)
    TSS = TPR - FPR
    N   = TP+FP+FN+TN
    E   = ((TP+FP)*(TP+FN)+(FP+TN)*(FN+TN))/(N+1e-9)
    HSS = (TP+TN-E)/(N-E+1e-9)
    try:    auc = roc_auc_score(y_true_binary, y_prob_mx)
    except: auc = float('nan')
    brier     = float(np.mean((y_prob_mx-y_true_binary)**2))
    clim      = y_true_binary.mean()
    brier_ref = clim*(1-clim)+1e-9
    bss       = 1.0 - brier/brier_ref
    FAR       = FP/(TP+FP+1e-9)
    if label:
        print(f'\n── {label} ──────────────────────────────────────')
    print(f'  Threshold : {threshold:.2f}')
    print(f'  TP:{TP:6d}  FP:{FP:8d}  FN:{FN:6d}  TN:{TN:8d}')
    print(f'  TSS  : {TSS:.4f}   (target >= 0.60)')
    print(f'  HSS  : {HSS:.4f}   (target >= 0.50)')
    print(f'  AUC  : {auc:.4f}   (target >= 0.85)')
    print(f'  BSS  : {bss:.4f}   (target >  0.10)')
    print(f'  FAR  : {FAR:.4f}   (target <  0.40)')
    print(f'  TPR  : {TPR:.4f}  |  FPR : {FPR:.4f}')
    return {'TSS':TSS,'HSS':HSS,'AUC':auc,'BSS':bss,
            'FAR':FAR,'TPR':TPR,'FPR':FPR,
            'TP':TP,'FP':FP,'FN':FN,'TN':TN}


# (duplicate sweep_threshold removed here — single 3-return version now lives in Cell 12 "Load Best Model")


# ── INPUT CLAMP CONSTANT ────────────────────────────────────────────────
# The derivative SHEF channels (dFs_dt, dFh_dt, d2Fs_dt2) have tiny scaler
# stds, so rare flare-spike windows reach |z| ~ 22-25 after standardisation.
# Clamping to ±10 removes the forward-pass overflow without touching the
# >99% of windows that sit well inside that range.
INPUT_CLAMP = 5.0


@torch.no_grad()
def run_inference(model_obj, loader):
    model_obj.eval()
    fore_l, now_l, ttps = [], [], []
    yf_all, ync_all, yttp_all = [], [], []
    for X_b, yf_b, ync_b, yttp_b in loader:
        X_b = torch.clamp(X_b.to(DEVICE), -INPUT_CLAMP, INPUT_CLAMP)   # ← clamp added
        X_b = torch.clamp(X_b.to(DEVICE), -5.0, 5.0)
        fl, nl, tp = model_obj(X_b)
        fore_l.append(fl.cpu()); now_l.append(nl.cpu()); ttps.append(tp.cpu())
        yf_all.append(yf_b); ync_all.append(ync_b); yttp_all.append(yttp_b)
    fore_probs = F.softmax(torch.cat(fore_l), dim=1).numpy()
    now_probs  = F.softmax(torch.cat(now_l),  dim=1).numpy()
    ttps       = torch.cat(ttps).squeeze(-1).numpy()
    return (fore_probs, now_probs, ttps,
            torch.cat(yf_all).numpy(),
            torch.cat(ync_all).numpy(),
            torch.cat(yttp_all).numpy())


@torch.no_grad()
def mc_dropout_predict(model_obj, X_tensor, n_passes=MC_PASSES):
    """Monte Carlo Dropout: enable dropout at inference for uncertainty."""
    model_obj.train()  # enables dropout
    all_probs = []
    X_tensor  = torch.clamp(X_tensor.to(DEVICE), -INPUT_CLAMP, INPUT_CLAMP)  # ← clamp added
    for _ in range(n_passes):
        fl, _, _ = model_obj(X_tensor)
        all_probs.append(F.softmax(fl, dim=1).cpu().numpy())
    model_obj.eval()
    all_probs  = np.stack(all_probs)  # (n_passes, N, 5)
    mean_probs = all_probs.mean(axis=0)
    p_mx       = mean_probs[:, 3] + mean_probs[:, 4]
    p_mx_all   = all_probs[:, :, 3] + all_probs[:, :, 4]
    ci_lo      = np.percentile(p_mx_all, 5,  axis=0)
    ci_hi      = np.percentile(p_mx_all, 95, axis=0)
    return mean_probs, p_mx, ci_lo, ci_hi


print('Evaluation utilities loaded.')
print(f'Input clamp active at ±{INPUT_CLAMP} (protects against derivative-channel spikes)')

Evaluation utilities loaded.
Input clamp active at ±5.0 (protects against derivative-channel spikes)


In [66]:
# ── Cell 11: Training Loop ─────────────────────────────────────────────────

best_val_tss = -1.0
best_epoch   = 0
patience_cnt = 0

CKPT_PATH_LOCAL = '/content/deepflare_best.pt'
CKPT_PATH_DRIVE = os.path.join(OUTPUT_DIR, 'deepflare_best.pt')

for _fname in ['deepflare_best.pt', 'deepflare_best (1).pt', 'deepflare_best (2).pt',
               'deepflare_best (3).pt', 'deepflare_best (4).pt', 'deepflare_best (5).pt']:
    _p = os.path.join(OUTPUT_DIR, _fname)
    if os.path.exists(_p):
        os.remove(_p)
        print(f'Deleted stale Drive checkpoint: {_fname}')
if os.path.exists(CKPT_PATH_LOCAL):
    os.remove(CKPT_PATH_LOCAL)
    print('Deleted stale local checkpoint.')

model = DeepFlarePredictor(
    n_features  = N_FEATURES,
    n_classes   = N_CLASSES,
    channels    = [N_FEATURES, 48, 96, 96, 160, 160],  # bumped — input grew 7->13ch
    dilations   = DILATIONS,
    kernel_size = KERNEL_SIZE,
    dropout     = DROPOUT
).to(DEVICE)
print('Training fresh from scratch.')

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)

WARMUP_EPOCHS = 5   # extended from 3 — prevents premature convergence

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return LR_MIN/LR + (1 - LR_MIN/LR) * 0.5 * (1 + math.cos(math.pi * progress))

scheduler   = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
history     = {k: [] for k in ['train_loss','val_loss','val_TSS','val_AUC','focal','tss','cal']}
INPUT_CLAMP = 5.0

print(f"\n{'='*65}")
print(f'Training DeepFlare Predictor | LR={LR} | patience={PATIENCE}')
print(f'  Device: {DEVICE} | Batch: {BATCH_SIZE} | No sampler (real M+ rate)')
print(f'  Input clamp: ±{INPUT_CLAMP} | Grad clip: max_norm=1.0, drop if >50')
print(f"{'='*65}\n")

for epoch in range(EPOCHS):
    model.train()
    epoch_loss  = 0.0
    comp_accum  = {'focal':0.0,'tss':0.0,'cal':0.0}
    n_batches   = 0
    nan_batches = 0

    for X_b, yf_b, ync_b, yttp_b in train_loader:
        X_b    = torch.clamp(X_b.to(DEVICE), -INPUT_CLAMP, INPUT_CLAMP)
        yf_b   = yf_b.to(DEVICE)
        ync_b  = ync_b.to(DEVICE)
        yttp_b = yttp_b.to(DEVICE).unsqueeze(1)

        optimizer.zero_grad()
        fore, now, ttp = model(X_b)

        fore = torch.clamp(fore, -20, 20)
        now  = torch.clamp(now,  -20, 20)
        ttp  = torch.clamp(ttp,  -100, 100)

        L_fore, comps = aspire_loss(fore, yf_b)
        L_now, _      = focal_only(now, ync_b)

        ttp_mask = (yttp_b > 0).float()
        L_ttp    = ttp_loss_fn(ttp*ttp_mask, yttp_b*ttp_mask) \
                   if ttp_mask.sum() > 0 else torch.tensor(0.0, device=DEVICE)

        loss = W_FORECAST*L_fore + W_NOWCAST*L_now + W_TTP*L_ttp

        if not torch.isfinite(loss):
            nan_batches += 1
            optimizer.zero_grad()
            continue

        loss.backward()

        # ── Fix: loosened gradient clip — was throttling learning ─────────
        total_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        if total_norm > 50:
            optimizer.zero_grad()
            nan_batches += 1
            continue

        optimizer.step()
        epoch_loss += loss.item()
        for k, v in comps.items(): comp_accum[k] += v
        n_batches  += 1

    if nan_batches > 0:
        print(f'  ⚠️  {nan_batches} bad batches skipped')

    if n_batches == 0:
        print(f'  ❌ Full NaN epoch — stopping'); break

    scheduler.step()
    avg_loss = epoch_loss / n_batches

    model.eval()
    fore_l, yf_all = [], []
    with torch.no_grad():
        for X_b, yf_b, ync_b, yttp_b in val_loader:
            X_b = torch.clamp(X_b.to(DEVICE), -INPUT_CLAMP, INPUT_CLAMP)
            fl, _, _ = model(X_b)
            fore_l.append(fl.cpu())
            yf_all.append(yf_b)
    fore_logits = torch.cat(fore_l)
    yf_val_np   = torch.cat(yf_all).numpy()
    fore_probs  = F.softmax(fore_logits, dim=1).numpy()

    if not np.isfinite(fore_probs).all():
        print(f'  ❌ NaN in val probs — skipping')
        patience_cnt += 1
        if patience_cnt >= PATIENCE: break
        continue

    y_val_bin = (yf_val_np >= 3).astype(int)
    p_mx_val  = fore_probs[:, 3] + fore_probs[:, 4]

    val_loss_t = F.nll_loss(
        torch.log(torch.FloatTensor(fore_probs) + 1e-9),
        torch.LongTensor(yf_val_np)).item()

    try:    val_auc = roc_auc_score(y_val_bin, p_mx_val)
    except: val_auc = float('nan')

    yp = (p_mx_val >= 0.5).astype(int)
    TP = int(((yp==1)&(y_val_bin==1)).sum())
    FP = int(((yp==1)&(y_val_bin==0)).sum())
    FN = int(((yp==0)&(y_val_bin==1)).sum())
    TN = int(((yp==0)&(y_val_bin==0)).sum())
    val_tss = TP/(TP+FN+1e-9) - FP/(FP+TN+1e-9)

    history['train_loss'].append(avg_loss)
    history['val_loss'].append(val_loss_t)
    history['val_TSS'].append(val_tss)
    history['val_AUC'].append(val_auc)
    for k in ['focal','tss','cal']:
        history[k].append(comp_accum[k]/n_batches)

    lr_now = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1:3d}/{EPOCHS} | loss={avg_loss:.4f} "
          f"(F={comp_accum['focal']/n_batches:.3f} "
          f"T={comp_accum['tss']/n_batches:.3f} "
          f"C={comp_accum['cal']/n_batches:.3f}) | "
          f"val_loss={val_loss_t:.4f} | val_TSS={val_tss:.4f} | "
          f"val_AUC={val_auc:.4f} | lr={lr_now:.2e}")

    if val_tss > best_val_tss:
        best_val_tss = val_tss
        best_epoch   = epoch + 1
        patience_cnt = 0

        if os.path.exists(CKPT_PATH_LOCAL):
            os.remove(CKPT_PATH_LOCAL)
        torch.save({
            'epoch'       : best_epoch,
            'model_state' : model.state_dict(),
            'optim_state' : optimizer.state_dict(),
            'val_TSS'     : val_tss,
            'val_AUC'     : val_auc,
            'history'     : history,
            'scaler_mean' : scaler.mean_,
            'scaler_std'  : scaler.scale_,
            'feature_cols': FEATURE_COLS,
            'config'      : {
                'WINDOW':WINDOW,'HORIZON':HORIZON,'N_CLASSES':N_CLASSES,
                'N_FEATURES':N_FEATURES,'DILATIONS':DILATIONS,
                'KERNEL_SIZE':KERNEL_SIZE,'DROPOUT':DROPOUT}
        }, CKPT_PATH_LOCAL)

        # ── Fix: Drive copy is now non-fatal — training continues even if
        # Drive disconnects mid-session ─────────────────────────────────
        import shutil
        drive_ok = False
        try:
            if os.path.exists(CKPT_PATH_DRIVE):
                os.remove(CKPT_PATH_DRIVE)
            shutil.copy2(CKPT_PATH_LOCAL, CKPT_PATH_DRIVE)
            drive_ok = os.path.exists(CKPT_PATH_DRIVE)
        except OSError as e:
            print(f'  ⚠️  Drive copy failed (continuing on local only): {e}')

        local_ok = os.path.exists(CKPT_PATH_LOCAL)
        print(f'  ✅ Best model saved (val_TSS={val_tss:.4f} @ epoch {best_epoch}) '
              f'| local={local_ok} | drive={drive_ok}')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch+1}')
            break

print(f'\nDone. Best val TSS={best_val_tss:.4f} at epoch {best_epoch}')
print(f'Local checkpoint : {os.path.exists(CKPT_PATH_LOCAL)}')
print(f'Drive checkpoint : {os.path.exists(CKPT_PATH_DRIVE)}')

Deleted stale local checkpoint.
Training fresh from scratch.

Training DeepFlare Predictor | LR=5e-05 | patience=15
  Device: cuda | Batch: 256 | No sampler (real M+ rate)
  Input clamp: ±5.0 | Grad clip: max_norm=1.0, drop if >50

Epoch   1/80 | loss=1.2146 (F=0.156 T=1.002 C=0.781) | val_loss=1.5375 | val_TSS=0.0000 | val_AUC=0.7847 | lr=2.00e-05
  ✅ Best model saved (val_TSS=0.0000 @ epoch 1) | local=True | drive=True
Epoch   2/80 | loss=1.0179 (F=0.127 T=0.947 C=0.476) | val_loss=0.9118 | val_TSS=0.0000 | val_AUC=0.8337 | lr=3.00e-05
Epoch   3/80 | loss=0.8285 (F=0.117 T=0.744 C=0.332) | val_loss=1.0285 | val_TSS=0.5406 | val_AUC=0.8581 | lr=4.00e-05
  ✅ Best model saved (val_TSS=0.5406 @ epoch 3) | local=True | drive=True
Epoch   4/80 | loss=0.6831 (F=0.116 T=0.631 C=0.350) | val_loss=1.0661 | val_TSS=0.5298 | val_AUC=0.8572 | lr=5.00e-05
Epoch   5/80 | loss=0.6547 (F=0.109 T=0.614 C=0.341) | val_loss=0.9573 | val_TSS=0.4663 | val_AUC=0.8556 | lr=5.00e-05
Epoch   6/80 | loss=0.647

In [68]:
# ── Cell 12: Load Best Model + Threshold Tuning on Val ────────────────────
ckpt = torch.load(CKPT_PATH_LOCAL, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()
best_epoch   = ckpt['epoch']
best_val_tss = ckpt['val_TSS']
history      = ckpt.get('history', history)
print(f'Loaded best model from epoch {best_epoch} (val_TSS={best_val_tss:.4f})')

fore_probs_val, _, _, yf_val_np, _, _ = run_inference(model, val_loader)
y_val_bin = (yf_val_np >= 3).astype(int)
p_mx_val  = fore_probs_val[:, 3] + fore_probs_val[:, 4]


def sweep_threshold(y_true, y_prob, thresholds=None, max_far=0.5):
    """Sweep thresholds, report both TSS-optimal and HSS-optimal,
    constrained to FAR <= max_far where possible.

    Thresholds are percentile-based on y_prob's own distribution rather than
    a fixed [0.05, 0.95] grid. With a 0.68% base rate, a well-calibrated
    classifier's probabilities cluster far below 0.05 — a fixed grid starting
    at 0.05 silently predicts "all negative" everywhere it checks, which looks
    like TSS=0 collapse but is actually a thresholding artifact, not a model
    failure. Percentile thresholds always land inside the actual data range,
    calibrated or raw, low base rate or not.
    """
    if thresholds is None:
        thresholds = np.unique(np.quantile(y_prob, np.linspace(0.0, 1.0, 500)))
    rows = []
    for th in thresholds:
        yp = (y_prob >= th).astype(int)
        TP = int(((yp==1)&(y_true==1)).sum())
        FP = int(((yp==1)&(y_true==0)).sum())
        FN = int(((yp==0)&(y_true==1)).sum())
        TN = int(((yp==0)&(y_true==0)).sum())
        TPR = TP/(TP+FN+1e-9); FPR = FP/(FP+TN+1e-9)
        TSS = TPR - FPR
        FAR = FP/(TP+FP+1e-9)
        N   = TP+FP+FN+TN
        E   = ((TP+FP)*(TP+FN)+(FP+TN)*(FN+TN))/(N+1e-9)
        HSS = (TP+TN-E)/(N-E+1e-9)
        rows.append({'threshold':th,'TSS':TSS,'HSS':HSS,'TPR':TPR,'FPR':FPR,'FAR':FAR})
    df    = pd.DataFrame(rows)
    valid = df[df['FAR'] <= max_far]
    pool  = valid if len(valid) > 0 else df

    best_tss = pool.loc[pool['TSS'].idxmax()]
    best_hss = pool.loc[pool['HSS'].idxmax()]

    print(f"Best-TSS (FAR<={max_far}): thr={best_tss['threshold']:.2f}  "
          f"TSS={best_tss['TSS']:.4f}  HSS={best_tss['HSS']:.4f}  FAR={best_tss['FAR']:.3f}")
    print(f"Best-HSS (FAR<={max_far}): thr={best_hss['threshold']:.2f}  "
          f"TSS={best_hss['TSS']:.4f}  HSS={best_hss['HSS']:.4f}  FAR={best_hss['FAR']:.3f}")

    return float(best_hss['threshold']), float(best_tss['threshold']), df


print('\nThreshold sweep on validation set:')
OPTIMAL_THRESHOLD_HSS, OPTIMAL_THRESHOLD_TSS, thresh_df = sweep_threshold(y_val_bin, p_mx_val)

# ── Default to the TSS-optimal threshold, not HSS-optimal ─────────────────
# HSS is numerically unstable at very low base rates (M/X ~0.68% here) — it
# can be maximized by a degenerate, extreme threshold (we saw thr=1.00 pick
# a corner solution: TSS collapsed from 0.61 -> 0.22 on val, 0.03 on test).
# TSS is the standard primary metric in the flare-forecasting literature
# precisely because it's insensitive to climatological/base-rate skew.
# Still report HSS at this threshold for completeness, just don't use it
# to choose the operating point.
OPTIMAL_THRESHOLD = OPTIMAL_THRESHOLD_TSS

np.save(os.path.join(OUTPUT_DIR, 'optimal_threshold.npy'),
        np.array([OPTIMAL_THRESHOLD]))

val_metrics = compute_metrics(y_val_bin, p_mx_val,
                               threshold=OPTIMAL_THRESHOLD,
                               label='VALIDATION SET (HSS-optimal threshold)')

Loaded best model from epoch 27 (val_TSS=0.5780)

Threshold sweep on validation set:
Best-TSS (FAR<=0.5): thr=0.45  TSS=0.6093  HSS=0.0432  FAR=0.968
Best-HSS (FAR<=0.5): thr=0.93  TSS=0.1599  HSS=0.1242  FAR=0.890

── VALIDATION SET (HSS-optimal threshold) ──────────────────────────────────────
  Threshold : 0.45
  TP:    76  FP:    2290  FN:    10  TN:    6056
  TSS  : 0.6093   (target >= 0.60)
  HSS  : 0.0432   (target >= 0.50)
  AUC  : 0.8609   (target >= 0.85)
  BSS  : -13.8074   (target >  0.10)
  FAR  : 0.9679   (target <  0.40)
  TPR  : 0.8837  |  FPR : 0.2744


In [69]:
# ── Cell 13: Test Set Evaluation ───────────────────────────────────────────
fore_probs_test, now_probs_test, ttps_test, \
    yf_test_np, ync_test_np, yttp_test_np = run_inference(model, test_loader)

y_test_bin = (yf_test_np >= 3).astype(int)
p_mx_test  = fore_probs_test[:, 3] + fore_probs_test[:, 4]

test_metrics = compute_metrics(y_test_bin, p_mx_test,
                                threshold=OPTIMAL_THRESHOLD,
                                label='TEST SET (HSS-optimal threshold)')

print(f'\nContext: M/X base rate = {y_test_bin.mean()*100:.2f}%')

print('\n── Class-wise test accuracy (nowcast head) ──')
y_now_pred     = now_probs_test.argmax(axis=1)
present_labels = sorted(np.unique(np.concatenate([ync_test_np, y_now_pred])))
present_names  = [CLASS_NAMES[i] for i in present_labels]
print(classification_report(ync_test_np, y_now_pred,
                             labels=present_labels,
                             target_names=present_names,
                             zero_division=0))


── TEST SET (HSS-optimal threshold) ──────────────────────────────────────
  Threshold : 0.45
  TP:    37  FP:    1558  FN:    21  TN:    6885
  TSS  : 0.4534   (target >= 0.60)
  HSS  : 0.0320   (target >= 0.50)
  AUC  : 0.8423   (target >= 0.85)
  BSS  : -13.8280   (target >  0.10)
  FAR  : 0.9768   (target <  0.40)
  TPR  : 0.6379  |  FPR : 0.1845

Context: M/X base rate = 0.68%

── Class-wise test accuracy (nowcast head) ──
              precision    recall  f1-score   support

        A/bg       0.88      1.00      0.94      1258
           B       0.98      0.72      0.83      3005
           C       0.85      0.97      0.90      4112
           M       0.49      0.67      0.57       126
           X       0.00      0.00      0.00         0

    accuracy                           0.88      8501
   macro avg       0.64      0.67      0.65      8501
weighted avg       0.90      0.88      0.88      8501



In [72]:
# ── Cell 13b: Platt-Scaling Recalibration — targets BSS directly ───────────
# Isotonic regression has too many free parameters for ~50-90 positive
# examples and collapses the probability output to a near-constant function
# (you saw this: optimal threshold collapsed to 0.05 with TSS=0.0000).
# Platt scaling fits a single 1-D logistic regression (2 parameters: a, b)
# on the raw score, which is far more stable with this little positive data.
from sklearn.linear_model import LogisticRegression

platt = LogisticRegression(C=1e6, solver='lbfgs')  # high C = minimal extra regularisation
platt.fit(p_mx_val.reshape(-1, 1), y_val_bin)

p_mx_test_cal = platt.predict_proba(p_mx_test.reshape(-1, 1))[:, 1]
p_mx_val_cal  = platt.predict_proba(p_mx_val.reshape(-1, 1))[:, 1]

def brier_bss(y_true, p):
    brier = np.mean((p - y_true) ** 2)
    clim  = y_true.mean()
    ref   = clim * (1 - clim) + 1e-9
    return 1.0 - brier/ref, brier

bss_before, brier_before = brier_bss(y_test_bin, p_mx_test)
bss_after,  brier_after  = brier_bss(y_test_bin, p_mx_test_cal)

print(f'BSS before recalibration: {bss_before:.4f}  (Brier={brier_before:.4f})')
print(f'BSS after  recalibration: {bss_after:.4f}  (Brier={brier_after:.4f})')

try:
    auc_after = roc_auc_score(y_test_bin, p_mx_test_cal)
    print(f'AUC after recalibration : {auc_after:.4f}  (rank-based — should match original AUC)')
except Exception:
    pass

# NOTE: sweep_threshold is the 3-return version from Cell 12 — make sure
# Cell 12 has been (re-)run in this kernel session before running this cell.
# Use TSS-optimal threshold here too — HSS keeps picking degenerate corner
# solutions at this base rate (see Cell 12 comment). Discard the HSS-optimal
# return value instead of the TSS one.
_, OPTIMAL_THRESHOLD_TSS_CAL, thresh_df_cal = sweep_threshold(y_val_bin, p_mx_val_cal)

test_metrics_recal = compute_metrics(y_test_bin, p_mx_test_cal,
                                      threshold=OPTIMAL_THRESHOLD_TSS_CAL,
                                      label='TEST SET (Platt-recalibrated, TSS-optimal threshold)')


BSS before recalibration: -13.8280  (Brier=0.1005)
BSS after  recalibration: 0.0072  (Brier=0.0067)
AUC after recalibration : 0.8423  (rank-based — should match original AUC)
Best-TSS (FAR<=0.5): thr=0.01  TSS=0.6093  HSS=0.0432  FAR=0.968
Best-HSS (FAR<=0.5): thr=0.07  TSS=0.1599  HSS=0.1242  FAR=0.890

── TEST SET (Platt-recalibrated, TSS-optimal threshold) ──────────────────────────────────────
  Threshold : 0.01
  TP:    37  FP:    1558  FN:    21  TN:    6885
  TSS  : 0.4534   (target >= 0.60)
  HSS  : 0.0320   (target >= 0.50)
  AUC  : 0.8423   (target >= 0.85)
  BSS  : 0.0072   (target >  0.10)
  FAR  : 0.9768   (target <  0.40)
  TPR  : 0.6379  |  FPR : 0.1845


In [70]:
# ── Cell 13c: Nowcast-Gated Forecast Score (post-hoc, no retraining) ───────
# Does NOT touch the trained model, the forecast probabilities used for your
# reported AUC, or anything upstream. This is purely a post-hoc combination
# you can compare against the raw forecast score. Idea: a forecast positive
# during a window the nowcast head confidently scores as quiet (A/bg) is
# almost certainly the kind of false alarm driving FAR up — gate the
# forecast probability down in those cases, leave it untouched otherwise.
# Real M/X buildup is rarely accompanied by a confident "quiet" nowcast, so
# true positives should mostly survive the gate while easy false positives
# get suppressed -> higher precision -> higher HSS at the same threshold.

p_now_quiet_val  = fore_probs_val[:, 0] * 0  # placeholder, overwritten below
# (re-run inference for nowcast on val if not already in memory)
_, now_probs_val_g, _, _, _, _ = run_inference(model, val_loader)
p_now_quiet_val  = now_probs_val_g[:, 0]            # P(currently A/bg) on val
p_now_quiet_test = now_probs_test[:, 0]             # P(currently A/bg) on test (already computed in Cell 13)

GATE_STRENGTH = 0.7   # 0 = no gating (identical to raw forecast), 1 = full suppression
                       # when nowcast is fully confident the window is quiet

gate_val  = 1.0 - GATE_STRENGTH * p_now_quiet_val
gate_test = 1.0 - GATE_STRENGTH * p_now_quiet_test

p_mx_val_gated  = p_mx_val  * gate_val
p_mx_test_gated = p_mx_test * gate_test

print(f'Gate strength: {GATE_STRENGTH}')
print(f'Mean gate value — val: {gate_val.mean():.3f} | test: {gate_test.mean():.3f}')

try:
    auc_gated_val  = roc_auc_score(y_val_bin,  p_mx_val_gated)
    auc_gated_test = roc_auc_score(y_test_bin, p_mx_test_gated)
    print(f'AUC (gated)  — val: {auc_gated_val:.4f} | test: {auc_gated_test:.4f}  '
          f'(compare to raw — val: {roc_auc_score(y_val_bin, p_mx_val):.4f} | '
          f'test: {roc_auc_score(y_test_bin, p_mx_test):.4f})')
except Exception as e:
    print(f'AUC compute failed: {e}')

# Re-sweep threshold on the GATED score (its scale differs from raw forecast)
_, OPTIMAL_THRESHOLD_TSS_GATED, thresh_df_gated = sweep_threshold(y_val_bin, p_mx_val_gated)

print('\n--- Comparison at TSS-optimal threshold ---')
print('RAW forecast:')
_ = compute_metrics(y_test_bin, p_mx_test, threshold=OPTIMAL_THRESHOLD,
                     label='TEST SET (raw forecast, TSS-optimal)')
print('\nNOWCAST-GATED forecast:')
_ = compute_metrics(y_test_bin, p_mx_test_gated, threshold=OPTIMAL_THRESHOLD_TSS_GATED,
                     label='TEST SET (nowcast-gated, TSS-optimal)')

print('\nIf HSS/FAR improved with TSS/AUC roughly unchanged or better, the gate is '
      'doing real work. If TSS dropped meaningfully, lower GATE_STRENGTH (e.g. 0.3-0.5) '
      'and re-run this cell only — no retraining needed.')


Gate strength: 0.7
Mean gate value — val: 0.878 | test: 0.881
AUC (gated)  — val: 0.8614 | test: 0.8424  (compare to raw — val: 0.8609 | test: 0.8423)
Best-TSS (FAR<=0.5): thr=0.44  TSS=0.6093  HSS=0.0432  FAR=0.968
Best-HSS (FAR<=0.5): thr=0.92  TSS=0.1696  HSS=0.1224  FAR=0.895

--- Comparison at TSS-optimal threshold ---
RAW forecast:

── TEST SET (raw forecast, TSS-optimal) ──────────────────────────────────────
  Threshold : 0.45
  TP:    37  FP:    1558  FN:    21  TN:    6885
  TSS  : 0.4534   (target >= 0.60)
  HSS  : 0.0320   (target >= 0.50)
  AUC  : 0.8423   (target >= 0.85)
  BSS  : -13.8280   (target >  0.10)
  FAR  : 0.9768   (target <  0.40)
  TPR  : 0.6379  |  FPR : 0.1845

NOWCAST-GATED forecast:

── TEST SET (nowcast-gated, TSS-optimal) ──────────────────────────────────────
  Threshold : 0.44
  TP:    37  FP:    1554  FN:    21  TN:    6889
  TSS  : 0.4539   (target >= 0.60)
  HSS  : 0.0321   (target >= 0.50)
  AUC  : 0.8424   (target >= 0.85)
  BSS  : -13.5343   (ta

In [ ]:
# ── Cell 14: MC Dropout Uncertainty ────────────────────────────────────────
print(f'\nRunning MC Dropout ({MC_PASSES} passes) on test set...')

N_MC       = min(2000, len(X_test_s))
X_mc       = torch.FloatTensor(X_test_s[:N_MC])
y_mc       = y_test_bin[:N_MC]

mean_probs_mc, p_mx_mc, ci_lo_mc, ci_hi_mc = mc_dropout_predict(model, X_mc)
ci_width = ci_hi_mc - ci_lo_mc

print(f'MC Dropout 90% CI width — mean: {ci_width.mean():.4f} | '
      f'median: {np.median(ci_width):.4f}')

p_mx_point = fore_probs_test[:N_MC, 3] + fore_probs_test[:N_MC, 4]
print(f'Correlation point vs MC mean: {np.corrcoef(p_mx_point, p_mx_mc)[0,1]:.4f}')

print('\nMC uncertainty by ground-truth class:')
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    mask = (yf_test_np[:N_MC] == cls_idx)
    if mask.sum() > 0:
        print(f'  {cls_name}: mean CI width = {ci_width[mask].mean():.4f}  '
              f'(n={mask.sum():,})')


Running MC Dropout (50 passes) on test set...
MC Dropout 90% CI width — mean: 0.3271 | median: 0.3563
Correlation point vs MC mean: 0.9400

MC uncertainty by ground-truth class:
  A/bg: mean CI width = 0.3189  (n=1,677)
  C: mean CI width = 0.3678  (n=311)
  M: mean CI width = 0.2620  (n=6)
  X: mean CI width = 0.5877  (n=6)


In [ ]:
# ── Cell 15: Baseline Comparison (lightweight — no RAM crash) ──────────────
# Uses summary features (mean/std/max per channel) instead of full flattened
# windows — avoids the RAM crash from flattening N*60*7 arrays

def extract_summary_features(X_arr):
    """Extract mean, std, max, min, last value, slope, max_deriv per channel."""
    N, T, C = X_arr.shape
    t_idx   = np.arange(T)
    feats   = []
    for i in range(N):
        row = []
        for c in range(C):
            ch = X_arr[i, :, c]
            row += [ch.mean(), ch.std(), ch.max(), ch.min(),
                    ch[-1],
                    np.polyfit(t_idx, ch, 1)[0],
                    np.abs(np.diff(ch)).max()]
        row.append(X_arr[i, :, 2].max())  # hardening peak
        feats.append(row)
    return np.array(feats, dtype=np.float32)

print('Extracting summary features for baselines...')
N_BASE       = min(50_000, len(X_train_s))
X_base_train = extract_summary_features(X_train_s[:N_BASE])
X_base_val   = extract_summary_features(X_val_s)
X_base_test  = extract_summary_features(X_test_s)
y_bin_train  = (y5_train[:N_BASE] >= 3).astype(int)
y_bin_val    = (y5_val   >= 3).astype(int)
y_bin_test   = (y5_test  >= 3).astype(int)
print(f'Feature shape: {X_base_train.shape}')

# Logistic Regression
print('\nFitting Logistic Regression...')
lr_model = Pipeline([('scaler', StandardScaler()),
                     ('clf', LogisticRegression(class_weight='balanced',
                                                max_iter=1000, C=0.1))])
lr_model.fit(X_base_train, y_bin_train)
p_mx_lr       = lr_model.predict_proba(X_base_test)[:, 1]
best_th_lr, _ = sweep_threshold(y_bin_val,
                                lr_model.predict_proba(X_base_val)[:, 1])
lr_metrics    = compute_metrics(y_bin_test, p_mx_lr,
                                threshold=best_th_lr,
                                label='BASELINE: Logistic Regression')

# XGBoost
xgb_metrics = None
if HAS_XGBOOST:
    print('\nFitting XGBoost...')
    scale_pw  = float((y_bin_train==0).sum()) / ((y_bin_train==1).sum()+1e-9)
    xgb_kwargs = dict(n_estimators=200, max_depth=5, learning_rate=0.05,
                      scale_pos_weight=scale_pw, subsample=0.8,
                      colsample_bytree=0.8, eval_metric='auc', verbosity=0)
    if DEVICE.type == 'cuda':
        xgb_kwargs.update(tree_method='hist', device='cuda')
    else:
        xgb_kwargs.update(tree_method='hist')
    xgb_model = XGBClassifier(**xgb_kwargs)
    xgb_model.fit(X_base_train, y_bin_train, verbose=False)
    p_mx_xgb       = xgb_model.predict_proba(X_base_test)[:, 1]
    best_th_xgb, _ = sweep_threshold(y_bin_val,
                                     xgb_model.predict_proba(X_base_val)[:, 1])
    xgb_metrics    = compute_metrics(y_bin_test, p_mx_xgb,
                                     threshold=best_th_xgb,
                                     label='BASELINE: XGBoost')

print(f"\n{'Model':<35} {'TSS':>8} {'AUC':>8} {'FAR':>8}")
print('─' * 55)
print(f"  {'DeepFlare TCN (ours)':<33} "
      f"{test_metrics['TSS']:8.4f} {test_metrics['AUC']:8.4f} {test_metrics['FAR']:8.4f}")
print(f"  {'Logistic Regression':<33} "
      f"{lr_metrics['TSS']:8.4f} {lr_metrics['AUC']:8.4f} {lr_metrics['FAR']:8.4f}")
if xgb_metrics:
    print(f"  {'XGBoost':<33} "
          f"{xgb_metrics['TSS']:8.4f} {xgb_metrics['AUC']:8.4f} {xgb_metrics['FAR']:8.4f}")

Extracting summary features for baselines...


KeyboardInterrupt: 

In [ ]:
# ── Cell 16: Visualisations ─────────────────────────────────────────────────
fig_dir = os.path.join(OUTPUT_DIR, 'figures')
os.makedirs(fig_dir, exist_ok=True)

# 1. Training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'], label='Train', lw=1.5)
axes[0].plot(history['val_loss'],   label='Val',   lw=1.5)
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['val_TSS'], 'b-o', markersize=3, lw=1.5, label='Val TSS')
axes[1].axhline(0.60, ls='--', color='r', alpha=0.7, label='Target=0.60')
axes[1].axvline(best_epoch-1, ls=':', color='g', label=f'Best (ep {best_epoch})')
axes[1].set_title('Val TSS'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(history['focal'], label='Focal', lw=1.5)
axes[2].plot(history['tss'],   label='TSS',   lw=1.5)
axes[2].plot(history['cal'],   label='Brier', lw=1.5)
axes[2].set_title('ASPIRE Components'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'training_curves.png'), dpi=150)
plt.close(); print('Saved training_curves.png')

# 2. Threshold ROC
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(thresh_df['FPR'], thresh_df['TPR'], 'b-o', markersize=4, lw=1.5)
best_row = thresh_df.loc[thresh_df['TSS'].idxmax()]
ax.plot(best_row['FPR'], best_row['TPR'], 'r*', markersize=16,
        label=f"Optimal theta={OPTIMAL_THRESHOLD:.2f} (TSS={best_row['TSS']:.3f})")
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('Threshold-ROC (Validation Set)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'threshold_roc.png'), dpi=150)
plt.close(); print('Saved threshold_roc.png')

# 3. Reliability diagram
bins = np.linspace(0, 1, 11)
frac_pos, mean_pred = [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (p_mx_test >= lo) & (p_mx_test < hi)
    if mask.sum() > 0:
        frac_pos.append(y_test_bin[mask].mean())
        mean_pred.append(p_mx_test[mask].mean())
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0,1],[0,1],'k--',lw=1,label='Perfect')
ax.plot(mean_pred, frac_pos,'s-',lw=2,markersize=8,label='DeepFlare')
ax.set_xlabel('Mean predicted P(M+)'); ax.set_ylabel('Fraction M+ events')
ax.set_title('Reliability Diagram'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'reliability_diagram.png'), dpi=150)
plt.close(); print('Saved reliability_diagram.png')

# 4. MC Dropout uncertainty
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(ci_width[y_mc==0], bins=50, alpha=0.6, label='No M+ flare',
             density=True, color='steelblue')
axes[0].hist(ci_width[y_mc==1], bins=50, alpha=0.6, label='M+ flare',
             density=True, color='firebrick')
axes[0].set_xlabel('90% CI width'); axes[0].set_title('MC Dropout Uncertainty')
axes[0].legend()
axes[1].scatter(p_mx_mc, ci_width, alpha=0.3, s=2, c=y_mc, cmap='RdYlGn_r')
axes[1].set_xlabel('Mean P(M+)'); axes[1].set_ylabel('90% CI width')
axes[1].set_title('P(M+) vs Uncertainty')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'mc_dropout_uncertainty.png'), dpi=150)
plt.close(); print('Saved mc_dropout_uncertainty.png')

Saved training_curves.png
Saved threshold_roc.png
Saved reliability_diagram.png
Saved mc_dropout_uncertainty.png


In [ ]:
# ── Cell 17: Ablation Study ─────────────────────────────────────────────────
def quick_ablation(name, X_tr, X_vl, y_tr, y_vl, epochs=15, patience=5):
    n_feat  = X_tr.shape[2]
    variant = DeepFlarePredictor(
        n_features=n_feat,
        channels=[n_feat, 32, 64, 64, 128, 128]
    ).to(DEVICE)
    opt     = torch.optim.Adam(variant.parameters(), lr=LR, weight_decay=1e-5)
    loss_fn = ASPIRELoss(CLASS_WEIGHTS, N_CLASSES, focal_gamma=FOCAL_GAMMA).to(DEVICE)
    tr_ds   = SolarFlareDataset(X_tr, y_tr, y_tr, np.zeros(len(y_tr), np.float32))
    tr_ld   = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True)
    vl_ds   = SolarFlareDataset(X_vl, y_vl, y_vl, np.zeros(len(y_vl), np.float32))
    vl_ld   = DataLoader(vl_ds, batch_size=BATCH_SIZE*2, shuffle=False)
    best_tss, wait = -1, 0
    for ep in range(epochs):
        variant.train()
        for X_b, yf_b, _, _ in tr_ld:
            X_b, yf_b = X_b.to(DEVICE), yf_b.to(DEVICE)
            opt.zero_grad()
            fl, _, _ = variant(X_b)
            fl = torch.clamp(fl, -20, 20)
            L, _ = loss_fn(fl, yf_b)
            if torch.isfinite(L):
                L.backward()
                nn.utils.clip_grad_norm_(variant.parameters(), 0.5)
                opt.step()
        fp, _, _, yf_np, _, _ = run_inference(variant, vl_ld)
        yb = (yf_np>=3).astype(int); pm = fp[:,3]+fp[:,4]
        yp = (pm>=0.5).astype(int)
        TP = int(((yp==1)&(yb==1)).sum()); FP = int(((yp==1)&(yb==0)).sum())
        FN = int(((yp==0)&(yb==1)).sum()); TN = int(((yp==0)&(yb==0)).sum())
        tss = TP/(TP+FN+1e-9) - FP/(FP+TN+1e-9)
        if tss > best_tss: best_tss, wait = tss, 0
        else:
            wait += 1
            if wait >= patience: break
    return best_tss

print(f"\n{'='*60}\nABLATION STUDY (15 epochs each)\n{'='*60}")
ablation_results = {}

print('A1: soft-only (1 channel)...')
ablation_results['Soft-only (1ch)'] = quick_ablation(
    'A1', X_train_s[:,:,:1], X_val_s[:,:,:1], y5_train, y5_val)
print(f"  TSS: {ablation_results['Soft-only (1ch)']:.4f}")

print('A2: soft+hard (2ch, no hardening ratio)...')
ablation_results['Soft+Hard (2ch)'] = quick_ablation(
    'A2', X_train_s[:,:,:2], X_val_s[:,:,:2], y5_train, y5_val)
print(f"  TSS: {ablation_results['Soft+Hard (2ch)']:.4f}")

print('A3: 3-channel core (soft+hard+ratio, no derivatives)...')
ablation_results['3-ch core'] = quick_ablation(
    'A3', X_train_s[:,:,:3], X_val_s[:,:,:3], y5_train, y5_val)
print(f"  TSS: {ablation_results['3-ch core']:.4f}")

ablation_results['Full 7-ch SHEF (ours)'] = best_val_tss

print(f"\n{'Model':<35} {'Val TSS':>10}")
print('─' * 48)
for k, v in ablation_results.items():
    marker = ' ← best' if v == max(ablation_results.values()) else ''
    print(f'  {k:<33} {v:10.4f}{marker}')


ABLATION STUDY (15 epochs each)
A1: soft-only (1 channel)...


KeyboardInterrupt: 

In [ ]:
# ── Cell 18: Save Full Results ──────────────────────────────────────────────
results_df = pd.DataFrame({
    'Metric'  : ['TSS','HSS','AUC-ROC','BSS','FAR','TPR','FPR',
                 'Best Epoch','Threshold','MC CI Width'],
    'Val'     : [val_metrics['TSS'],  val_metrics['HSS'],
                 val_metrics['AUC'],  val_metrics['BSS'],
                 val_metrics['FAR'],  val_metrics['TPR'],
                 val_metrics['FPR'],  best_epoch,
                 OPTIMAL_THRESHOLD,   ci_width.mean()],
    'Test'    : [test_metrics['TSS'], test_metrics['HSS'],
                 test_metrics['AUC'], test_metrics['BSS'],
                 test_metrics['FAR'], test_metrics['TPR'],
                 test_metrics['FPR'], best_epoch,
                 OPTIMAL_THRESHOLD,   ci_width.mean()],
    'Target'  : ['>= 0.60','>= 0.50','>= 0.85','> 0.10',
                 '< 0.40','—','—','—','—','—']
})

results_path = os.path.join(OUTPUT_DIR, 'solfpred_results.csv')
results_df.to_csv(results_path, index=False)
print(f'Results saved to {results_path}')
print(f"\n{'='*65}")
print('SoLFPRED — FINAL RESULTS')
print(f"{'='*65}")
print(results_df.to_string(index=False))
print(f'\nCheckpoint : {CKPT_PATH}')
print(f"{'='*65}")

Results saved to /content/drive/MyDrive/BAH/isro_hackathon_data/forecasting_outputs/solfpred_results.csv

SoLFPRED — FINAL RESULTS
     Metric        Val       Test  Target
        TSS   0.580151   0.067926 >= 0.60
        HSS   0.024202   0.001365 >= 0.50
    AUC-ROC   0.864862   0.605719 >= 0.85
        BSS -38.923576 -87.219571  > 0.10
        FAR   0.981668   0.990149  < 0.40
        TPR   0.865804   0.979946       —
        FPR   0.285653   0.912020       —
 Best Epoch   1.000000   1.000000       —
  Threshold   0.580000   0.580000       —
MC CI Width   0.327098   0.327098       —

Checkpoint : /content/drive/MyDrive/BAH/isro_hackathon_data/forecasting_outputs/deepflare_best.pt
